# 20.1 ML system design

A production ML system is a set of tradeoffs: model quality matters only when latency, reliability, and ownership let the prediction arrive safely.

Supervised learning metrics feed the quality term, while deployment lessons turn those metrics into service-level constraints.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(201)


def latency_summary(latency_ms):
    return {
        "p50_ms": float(np.percentile(latency_ms, 50)),
        "p95_ms": float(np.percentile(latency_ms, 95)),
        "mean_ms": float(np.mean(latency_ms)),
    }


def ops_summary(workload):
    seconds = max(float(workload["duration_seconds"]), 1.0)
    request_count = int(workload["latency_ms"].size)
    return {
        "name": workload["name"],
        "requests": request_count,
        "p50_ms": float(np.percentile(workload["latency_ms"], 50)),
        "p95_ms": float(np.percentile(workload["latency_ms"], 95)),
        "throughput_qps": request_count / seconds,
        "cost_dollars": float(np.sum(workload["cost_usd"])),
        "drift_stat": float(workload["drift_stat"]),
    }


def simulate_system_workload(name, request_count, base_qps, burst=False, degraded=False, diurnal=False, fanout=3):
    seconds = max(int(np.ceil(request_count / base_qps)), 1)
    arrivals = np.linspace(0.0, seconds, request_count, endpoint=False)
    load_multiplier = np.ones(request_count)
    if diurnal:
        phase = 2.0 * np.pi * arrivals / max(seconds, 1)
        load_multiplier = 1.0 + 0.35 * np.sin(phase)
    if burst:
        burst_start = request_count // 2
        burst_end = min(request_count, burst_start + max(request_count // 10, 1))
        load_multiplier[burst_start:burst_end] += 1.25
    feature_latency = RNG.lognormal(mean=np.log(35.0), sigma=0.18, size=request_count)
    model_latency = RNG.lognormal(mean=np.log(55.0), sigma=0.22, size=request_count)
    post_latency = RNG.lognormal(mean=np.log(20.0), sigma=0.16, size=request_count)
    dependency_latency = RNG.lognormal(mean=np.log(8.0 * fanout), sigma=0.20, size=request_count)
    latency_ms = feature_latency + model_latency + post_latency + dependency_latency
    latency_ms = latency_ms * load_multiplier
    if degraded:
        tail_mask = RNG.random(request_count) < 0.08
        latency_ms[tail_mask] += RNG.lognormal(mean=np.log(180.0), sigma=0.35, size=int(np.sum(tail_mask)))
    cost_usd = 0.00002 * fanout + latency_ms * 0.00000008
    drift_stat = abs(float(np.mean(latency_ms[: max(request_count // 5, 1)]) - np.mean(latency_ms[-max(request_count // 5, 1):])))
    return {
        "name": name,
        "latency_ms": latency_ms,
        "cost_usd": cost_usd,
        "duration_seconds": seconds,
        "drift_stat": drift_stat,
        "load_qps": base_qps,
    }


def make_workload_ladder():
    d1 = {
        "name": "D1 tiny request trace",
        "latency_ms": np.array([96, 102, 107, 111, 115, 119, 123, 129, 136, 142], dtype=float),
        "cost_usd": np.array([0.00008] * 10, dtype=float),
        "duration_seconds": 10.0,
        "drift_stat": 0.0,
        "load_qps": 1.0,
    }
    d2 = simulate_system_workload("D2 larger clean volume", 1_000, 50, fanout=2)
    d3 = simulate_system_workload("D3 burst plus degraded dependency", 20_000, 120, burst=True, degraded=True, fanout=3)
    d4 = simulate_system_workload("D4 real-ish diurnal trace", 120_000, 250, burst=True, diurnal=True, fanout=4)
    d5 = simulate_system_workload("D5 production-scale fanout simulation", 1_000_000, 850, burst=True, degraded=True, diurnal=True, fanout=6)
    return [d1, d2, d3, d4, d5]


## The concept, built once: weighted design score

The lesson's design score is $S=\sum_j w_jm_j$. We first write the reusable scorer and assert the exact traffic and dependency arithmetic from the plan: $10{,}000\times0.30=3{,}000$ and $0.995\times0.990=0.98505$.

In [ ]:

def score_design(metrics, weights):
    metric_values = np.array([metrics[key] for key in weights], dtype=float)
    weight_values = np.array([weights[key] for key in weights], dtype=float)
    if not np.isclose(np.sum(weight_values), 1.0):
        raise ValueError("weights must sum to 1")
    return float(np.sum(metric_values * weight_values))

traffic_split = 10_000 * 0.30
chain_availability = 0.995 * 0.990
assert traffic_split == 3_000
assert np.isclose(chain_availability, 0.98505)
print("candidate calls", traffic_split)
print("two-service availability", chain_availability)


Now plug in the lesson utility numbers. The same function evaluates a tiny D1 design trace while the assertions anchor the notebook to the lesson: $0.50\times0.82+0.30\times0.73+0.20\times0.90=0.809$.

In [ ]:

weights = {"quality": 0.50, "latency": 0.30, "reliability": 0.20}
metrics = {"quality": 0.82, "latency": 0.73, "reliability": 0.90}
design_score = score_design(metrics, weights)
assert np.isclose(design_score, 0.809)
print("weighted design score", round(design_score, 3))


## The dataset ladder

Family F17 has no shared ML-training ladder here. We build the operational workload ladder inline: D1 tiny trace, D2 larger volume, D3 spikes or drift, D4 real-ish synthetic trace, and D5 production-scale simulation.

In [ ]:

workloads = make_workload_ladder()
for workload in workloads:
    print(workload["name"])
    print("shape", workload["latency_ms"].shape)
    print("first latencies", np.round(workload["latency_ms"][:5], 2))


## Run the same method across D1-D5

The operational metric for this topic is p95 end-to-end latency. We also account for p50, throughput, cost, and a simple latency drift statistic so the table reflects real launch-review tradeoffs.

In [ ]:

workloads = make_workload_ladder()
summaries = [ops_summary(workload) for workload in workloads]

header = f"{'rung':<42} {'load':>10} {'p50':>9} {'p95':>9} {'cost':>10} {'qps':>10} {'p95_ms':>12}"
print(header)
for row in summaries:
    line = f"{row['name']:<42} {row['requests']:>10} {row['p50_ms']:>9.2f} {row['p95_ms']:>9.2f} {row['cost_dollars']:>10.3f} {row['throughput_qps']:>10.2f} {row['p95_ms']:>12.5f}"
    print(line)


The first row is the operational artifact: p95 latency, total cost, and throughput by rung. The second plot is the lesson metric versus load.

In [ ]:

names = [row["name"].split()[0] for row in summaries]
loads = np.array([row["requests"] for row in summaries], dtype=float)
p95_values = np.array([row["p95_ms"] for row in summaries], dtype=float)
cost_values = np.array([row["cost_dollars"] for row in summaries], dtype=float)
throughput_values = np.array([row["throughput_qps"] for row in summaries], dtype=float)
metric_values = np.array([row["p95_ms"] for row in summaries], dtype=float)
cost_per_request = cost_values / loads
normalized_p95 = p95_values / max(float(np.max(p95_values)), 1.0)
normalized_cost = cost_per_request / max(float(np.max(cost_per_request)), 1.0)
normalized_throughput = throughput_values / max(float(np.max(throughput_values)), 1.0)

fig = plt.figure(figsize=(15, 7))
grid = fig.add_gridspec(2, 5, height_ratios=[1.0, 1.15])
for index, name in enumerate(names):
    ax = fig.add_subplot(grid[0, index])
    values = [normalized_p95[index], normalized_cost[index], normalized_throughput[index]]
    ax.bar(["p95", "cost", "qps"], values)
    ax.set_ylim(0.0, 1.05)
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel("normalized")
summary_ax = fig.add_subplot(grid[1, :])
summary_ax.plot(loads, metric_values, marker="o")
summary_ax.set_xscale("log")
summary_ax.set_title("p95 latency vs load")
summary_ax.set_xlabel("records or requests")
summary_ax.set_ylabel("p95 latency")
fig.suptitle("Per-workload latency, cost, throughput, and metric-vs-load")
fig.tight_layout()


## Pitfall on D5: budgeting averages instead of tails

A design can have a mean under the $150$ ms budget while p95 violates the promise. The fix is to choose the candidate with tail headroom, not the one with the lowest average only.

In [ ]:

d5 = workloads[-1]
mean_latency = float(np.mean(d5["latency_ms"]))
p95_latency = float(np.percentile(d5["latency_ms"], 95))
budget_ms = 150.0
wrong_passes = mean_latency < budget_ms
right_passes = p95_latency < budget_ms
candidate_tail = d5["latency_ms"] * 0.72
fixed_p95 = float(np.percentile(candidate_tail, 95))
print("mean under budget", wrong_passes, round(mean_latency, 2))
print("p95 under budget", right_passes, round(p95_latency, 2))
print("fixed p95", round(fixed_p95, 2))


## Evaluate it + Practice

- Metric: p95 latency; no-skill baseline is choosing the highest offline quality design and ignoring service terms.
- Sanity check: D1 p95 should be close to the hand trace tail.
- Ablation: remove reliability or latency weights and watch the selected design become operationally fragile.
- Failure signals: p95 grows faster than load, drift rises across the day, or cost grows superlinearly.

Practice: change the candidate traffic split and recompute candidate calls.

Practice: add a third dependency with availability $0.992$ and recompute chain availability.

Practice: lower the D5 fanout and compare p95 and cost.